# 2 · Label the questions with the IPC assistant (Qwen2.5-7B)
Settings: **Accelerator: GPU T4 x2**, **Internet: On**, **Add-ons → Secrets → HF_TOKEN** (write token).
Inputs: your code dataset **and** the output of notebook 1. Run the trial cell interactively once, then *Save Version → Save & Run All* for the full run (up to 12 h, in the background).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
# shard of the judgments for THIS run: '0/1' = all; '0/2' then '1/2' = two halves (two runs or two notebooks)
SHARD = '0/1'

In [ ]:
import glob, os, shutil
src = os.path.dirname(glob.glob("/kaggle/input/**/make_questions.py", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/ipc", dirs_exist_ok=True, ignore=shutil.ignore_patterns(".venv", ".cache", "__pycache__", "models"))
%cd /kaggle/working/ipc
print("code from", src)

In [ ]:
!pip install -q "transformers>=4.56,<5" peft accelerate "bitsandbytes>=0.45" huggingface_hub pymupdf

In [ ]:
# models: your trained model + Llama-3.2-3B base + InLegalBERT (token needs access to meta-llama), and the reranker
!python download_model.py --repo gogulakrishnan256/hyena-llama-legal-v7
!python download_reranker.py

In [ ]:
# Ollama on GPU 1, the assistant on GPU 0
!apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1', 'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0', 'OLLAMA_KEEP_ALIVE': '-1'}
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b

In [ ]:
import glob
QA = os.path.dirname(glob.glob('/kaggle/input/**/questions.jsonl', recursive=True)[0])
os.environ.update({'CUDA_VISIBLE_DEVICES': '0', 'IPC_KEEP_RESIDENT': '1'})
print(QA)

In [ ]:
# TRIAL (2 judgments, ~3 min): check the numbers look sane, then you may skip this cell in the full run
!python label_ipc.py $QA/questions.jsonl --pdfs $QA/pdfs --out /kaggle/working/trial.jsonl --limit 2
!head -c 1500 /kaggle/working/trial.jsonl

In [ ]:
# FULL RUN: resumable, backed up to your private HF dataset <user>/ipc-router-labels every 50 questions
!python label_ipc.py $QA/questions.jsonl --pdfs $QA/pdfs --shard $SHARD --checkpoint --out /kaggle/working/labels/ipc_qwen2.5-7b_shard{SHARD.replace('/', 'of')}.jsonl